## Text input

In [1]:
%run init_env.py

Added to Python path: C:\git\cs5305
Environment initialization completed successfully.


In [2]:
llm = create_azure_llm()
agent = create_agent(llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


In [3]:
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Welcome to Selene Prime, the dazzling capital city of The Moon!

Nestled within the vast expanse of the Sea of Tranquility, Selene Prime is a marvel of human ingenuity and lunar adaptation. The city is housed inside a massive transparent dome made from ultra-durable, self-healing lunar glass that filters harmful cosmic rays while allowing breathtaking views of Earth and the star-studded cosmos.

Selene Prime's architecture blends sleek, futuristic designs with organic lunar rock formations, creating a harmonious balance between technology and the Moon’s natural landscape. The city is divided into interconnected sectors:

- **Helios Hub**: The political and administrative center where the Lunar Council governs with representatives from Earth and Moon colonies.

- **LunaTech District**: A bustling area filled with advanced research labs, robotics factories, and innovation centers focused on space technology and terraforming projects.

- **Orbital Gardens**: A vast network of hydroponic f

## Image input

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)
print(uploader.value)

FileUpload(value=(), accept='.png', description='Upload')

()


In [ ]:
import base64
img_b64 = None

if uploader.value:       # Get the first (and only) uploaded file dict
    uploaded_file = uploader.value[0]

    # This is a memoryview
    content_mv = uploaded_file["content"]

    # Convert memoryview -> bytes
    img_bytes = bytes(content_mv)  # or content_mv.tobytes()

    # Now base64 encode
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    print("Image successfully encoded to base64.")


In [21]:
if img_b64:    
    multimodal_question = HumanMessage(content=[
        {"type": "text", "text": "Tell me about this diagram"},
        {"type": "image", "base64": img_b64, "mime_type": "image/png"}
    ])

    response = agent.invoke(
        {"messages": [multimodal_question]}
    )

    print(response['messages'][-1].content)

This diagram represents the distribution and analysis of student grades for a class of 94 students. Here's a detailed breakdown:

1. Grade Distribution (Top Left Bar Chart):
- The vertical bar chart shows the number of students achieving each grade.
- Grades range from A+ to F.
- The counts for each grade are shown on top of each bar:
  - A+: 2 students
  - A: 11 students
  - A-: 15 students
  - B+: 8 students
  - B: 16 students
  - B-: 6 students
  - C+: 9 students
  - C: 6 students
  - D: 4 students
  - F: 3 students

2. Stacked Area Chart (Top Right):
- This chart visualizes the cumulative distribution or performance across the grade spectrum, with color-coded bands corresponding to different grade categories.
- The green area represents the highest grades (A+ to A-), followed by blue for B range, yellow for C range, orange for D, and red for F.
- It visually emphasizes the proportion of students in each grade category.

3. Grade Cutoffs Table (Bottom Left):
- This table compares th

In [32]:
if img_b64:   
    print("\n✅ Multimodal response metadata:") 
    pprint(response['messages'][-1].response_metadata)


✅ Multimodal response metadata:
{'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'},
                            'protected_material_code': {'detected': False,
                                                        'filtered': False},
                            'protected_material_text': {'detected': False,
                                                        'filtered': False},
                            'self_harm': {'filtered': False,
                                          'severity': 'safe'},
                            'sexual': {'filtered': False, 'severity': 'safe'},
                            'violence': {'filtered': False,
                                         'severity': 'safe'}},
 'finish_reason': 'stop',
 'id': 'chatcmpl-DSSlUgdl2F6b4K91qTEEdIa7KPBj6',
 'logprobs': None,
 'model_name': 'gpt-4.1-mini-2025-04-14',
 'model_provider': 'openai',
 'prompt_filter_results': [{'content_filter_results': {'hate': {'filtered': False,
              

In [15]:
# Check what deployments you have available
import os
from dotenv import load_dotenv

load_dotenv()

print("🔍 Current Azure AI Foundry Configuration:")
print("=" * 50)
print(f"Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Chat Deployment: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}")
print(f"Audio Deployment: {os.getenv('AZURE_OPENAI_AUDIO_DEPLOYMENT')}")
print(f"API Version: {os.getenv('AZURE_OPENAI_API_VERSION')}")
print("=" * 50)

# Test if we can create connections to both deployments
print("\n📋 Testing Model Availability:")
print("-" * 30)

try:
    from azure_openai_llm import create_azure_llm
    
    print("✅ Chat model connection:")
    chat_llm = create_azure_llm("chat")
    print(f"   Model type: {type(chat_llm).__name__}")
    
    print("\n✅ Audio model connection:")  
    audio_llm = create_azure_llm("audio")
    print(f"   Model type: {type(audio_llm).__name__}")
    
except Exception as e:
    print(f"❌ Error: {e}")

🔍 Current Azure AI Foundry Configuration:
Endpoint: https://aoai-foundry-swc.openai.azure.com/
Chat Deployment: lums-gpt-4.1-mini
Audio Deployment: lums-gpt-audio-mini
API Version: 2025-01-01-preview

📋 Testing Model Availability:
------------------------------
✅ Chat model connection:
Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview
   Model type: AzureChatOpenAI

✅ Audio model connection:
Using audio deployment: lums-gpt-audio-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview
   Model type: AzureChatOpenAI


In [38]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm # tqdm provides a visual progress bar during the recording loop.

# Steps:
# 2) Record audio from the microphone using sounddevice (sd).
# 3) Display a short progress bar during recording.
# 4) Wait for recording completion to ensure data is finalized.
# 5) Convert WAV bytes to Base64 encoded string.

# Recording settings
duration = 3  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")
print("\n✅ Audio successfully recorded and encoded to base64.")

Recording...


100%|██████████| 30/30 [00:03<00:00,  9.78it/s]


Done.

✅ Audio successfully recorded and encoded to base64.


In [39]:
if aud_b64:
    agent = create_agent(create_azure_llm("audio"))

    multimodal_question = HumanMessage(content=[
        {"type": "text", "text": "Follow the instructions in the audio and reply"},
        {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
    ])

    response = agent.invoke(
        {"messages": [multimodal_question]}
    )

    print(response['messages'][-1].content)

Using audio deployment: lums-gpt-audio-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview
Good afternoon and good night! How can I assist you further?
